# Stage13: Productization

This notebook verifies the saved SPY risk-alert artifact and its Flask API. The service exposes an end-of-day decision-support probability; it does not place trades or refit the model.

In [1]:
from pathlib import Path
import os
import subprocess
import sys
import time

import requests

if Path.cwd().name == "notebooks":
    os.chdir("..")
ROOT = Path.cwd()
if not (ROOT / "app.py").is_file():
    for candidate in (ROOT, *ROOT.parents):
        if (candidate / "project" / "app.py").is_file():
            ROOT = candidate / "project"
            break
model_path = ROOT / "model" / "spy_risk_alert_model.pkl"
assert model_path.is_file(), "Run the Stage13 pipeline cell before API verification."
print("Saved model:", model_path)

Saved model: /Users/cengchengyu/Documents/NYU/Class/Boot Camp 4/CS HW/bootcamp_chengyu_zeng/project/model/spy_risk_alert_model.pkl


## API verification with requests

The server is started only for this test, then shut down. A valid named-feature payload proves prediction works; an incomplete payload proves invalid input returns JSON HTTP 400.

In [2]:
from src.productization import load_model_artifact

artifact = load_model_artifact(model_path)
features = {name: 0.0 for name in artifact["feature_columns"]}
server = subprocess.Popen(
    [sys.executable, "app.py"], cwd=ROOT, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
base = "http://127.0.0.1:5050"
try:
    for _ in range(20):
        try:
            health = requests.get(base + "/health", timeout=0.5)
            if health.ok:
                break
        except requests.ConnectionError:
            time.sleep(0.2)
    else:
        raise RuntimeError("API did not start on port 5000")
    valid = requests.post(base + "/predict", json={"features": features}, timeout=5)
    invalid = requests.post(base + "/predict", json={"features": {}}, timeout=5)
    print("GET /health:", health.status_code, health.json())
    print("POST /predict:", valid.status_code, valid.json())
    print("POST /predict (bad input):", invalid.status_code, invalid.json())
    assert health.status_code == valid.status_code == 200
    assert invalid.status_code == 400 and "error" in invalid.json()
finally:
    server.terminate()
    server.wait(timeout=5)

GET /health: 200 {'feature_count': 13, 'status': 'ok', 'training_end': '2022-09-06'}
POST /predict: 200 {'alert': False, 'cutoff': 0.5199999999999999, 'event_threshold': 0.01699042268741321, 'probability': 0.0743888022491789, 'training_end': '2022-09-06'}
POST /predict (bad input): 400 {'error': "feature keys must match the model schema (missing: ['abs_return_t', 'intraday_range_t', 'log_volume_change_t', 'momentum_5_t', 'return_lag_1_t', 'rolling_abs_return_mean_5_t', 'rolling_return_std_10_t', 'rolling_volatility_5_t', 'weekday_Friday', 'weekday_Monday', 'weekday_Thursday', 'weekday_Tuesday', 'weekday_Wednesday'])"}
